In [ ]:

import kagglehub
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np
import re
import nltk

# Ignore unnecessary warnings
import warnings
warnings.filterwarnings("ignore")
print("Libraries imported successfully!")

#Install kagglehu
!pip install -q kagglehub

#download the dataset from kaggle
path = kagglehub.dataset_download("lburleigh/asap-2-0")

print("Dataset downloaded to:", path)

#show the files in the kaggle asap_2.0 dataset
print(os.listdir(path))

#loading the dataset to google colab
df = pd.read_csv(os.path.join(path, "ASAP2_train_sourcetexts.csv"))

Libraries imported successfully!


Using Colab cache for faster access to the 'asap-2-0' dataset.


In [ ]:

!pip install textstat language-tool-python scikit-learn nltk

In [ ]:

print(df.head())
print()
print(df.info())
print()
print(df.describe())
print()
print(df.columns)
print()
print(df.isnull().sum())
print()
#print(df.duplicated().sum())

In [ ]:

score_counts = df["score"].value_counts().sort_index()

plt.figure(figsize=(7,4))
plt.bar(score_counts.index.astype(str), score_counts.values, color="blue", edgecolor="black")
plt.xlabel("Score")
plt.ylabel("Number of essays")
plt.title("Distribution of essay scores")
for i, v in enumerate(score_counts.values):
    plt.text(i, v + max(score_counts.values)*0.01, str(v), ha="center", fontsize=9)
plt.tight_layout()
plt.show()

print(score_counts)
print()
print("Most common score is", score_counts.idxmax(), f"({score_counts.max()} essays,",
      f"{100*score_counts.max()/len(df):.1f}% of the dataset)")
print("Rarest score is", score_counts.idxmin(), f"({score_counts.min()} essays,",
      f"{100*score_counts.min()/len(df):.1f}% of the dataset)")

In [ ]:

prompt_counts = df["prompt_name"].value_counts()

plt.figure(figsize=(9,5))
plt.barh(prompt_counts.index[::-1], prompt_counts.values[::-1], color="green", edgecolor="black")
plt.xlabel("Number of essays")
plt.title("Essays per prompt")
plt.tight_layout()
plt.show()

print(f"{df['prompt_name'].nunique()} distinct prompts")

# Does average score vary a lot by prompt? If so, score isn't purely comparable across prompts.
score_by_prompt = df.groupby("prompt_name")["score"].agg(["mean", "std", "count"]).sort_values("mean")
score_by_prompt

In [ ]:

raw_word_count = df["full_text"].apply(lambda x: len(x.split()))

plt.figure(figsize=(8,5))
plt.hist(raw_word_count, bins=40, color="yellow", edgecolor="black")
plt.xlabel("Word count")
plt.ylabel("Number of essays")
plt.title("Distribution of essay length (raw word count)")
plt.tight_layout()
plt.show()

print(raw_word_count.describe())

In [ ]:

columns_to_keep = ["essay_id",
    "score",
    "full_text",
    "assignment",
    "prompt_name"]

df = df[columns_to_keep].copy()

df["full_text"] = df["full_text"].str.strip()
df["assignment"] = df["assignment"].str.strip()
df["prompt_name"] = df["prompt_name"].str.strip()

print((df["full_text"] == "").sum())
df["score"].value_counts().sort_index()

In [ ]:

#Feature engineering
df["word_count"] = df["full_text"].apply(lambda x: len(x.split()))
df["word_count"].describe()

In [ ]:

df[df["word_count"] < 20]

df.to_csv("cleaned_asap.csv", index=False)

In [ ]:

df["character_count"] = df["full_text"].apply(len)

df["sentence_count"] = df["full_text"].apply(lambda x: len(re.findall(r"[.!?]+", x))).replace(0, 1)

df["avg_sentence_length"] = (df["word_count"] / df["sentence_count"])

df["unique_words"] = df["full_text"].apply(lambda x: len(set(x.lower().split())))

In [ ]:

df["paragraph_count"] = df["full_text"].apply(
    lambda x: len(re.split(r"\n\s*\n", str(x).strip())))

df["avg_word_length"] = (df["character_count"] / df["word_count"])

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

df["stopword_ratio"] = df["full_text"].apply(
    lambda x: sum(1 for w in str(x).lower().split() if w in ENGLISH_STOP_WORDS) / len(str(x).split()))

df["avg_paragraph_len"] = (df["sentence_count"] / df["paragraph_count"])

# Type-Token Ratio (TTR) — measures how diverse the vocabulary is in an essay
df["type_token_ratio"] = df.apply(
    lambda row: row["unique_words"] / row["word_count"]
    if row["word_count"] != 0 else 0,
    axis=1)

df["type_token_ratio"].describe()

In [ ]:

!pip install -q textstat
import textstat

# Flesch Reading Ease
df["flesch_reading_ease"] = df["full_text"].apply(textstat.flesch_reading_ease)

# Flesch-Kincaid Grade Level
df["flesch_kincaid_grade"] = df["full_text"].apply(textstat.flesch_kincaid_grade)

df[["flesch_reading_ease", "flesch_kincaid_grade"]].describe()

In [ ]:

df["gunning_fog_score"] = df["full_text"].apply(lambda t: textstat.gunning_fog(t))
df["dale_chall_readability_score"] = df["full_text"].apply(lambda t: textstat.dale_chall_readability_score(t))

df["guiraud_index"] = df["unique_words"] / np.sqrt(df["word_count"].replace(0, np.nan))
df["guiraud_index"] = df["guiraud_index"].fillna(0)

df["long_word_ratio"] = df["full_text"].apply(
    lambda x: sum(1 for w in str(x).split() if len(w) > 6) / max(len(str(x).split()), 1))

CONJUNCTIONS = {'and', 'but', 'or', 'nor', 'for', 'yet', 'so', 'however', 'therefore', 'although'}
df["conjunction_ratio"] = df["full_text"].apply(
    lambda x: sum(1 for w in str(x).lower().split() if w in CONJUNCTIONS) / max(len(str(x).split()), 1))

df[["gunning_fog_score", "dale_chall_readability_score", "guiraud_index", "long_word_ratio", "conjunction_ratio"]].describe()

In [ ]:

# Everything the models train on — all cheap, all computed above, none of it needs spaCy/YAKE/LanguageTool
fast_features = [
    "word_count",
    "sentence_count",
    "character_count",
    "paragraph_count",
    "avg_word_length",
    "avg_sentence_length",
    "unique_words",
    "type_token_ratio",
    "stopword_ratio",
    "avg_paragraph_len",
    "flesch_reading_ease",
    "flesch_kincaid_grade",
    "gunning_fog_score",
    "dale_chall_readability_score",
    "guiraud_index",
    "long_word_ratio",
    "conjunction_ratio",
]

fast_features_df = df[fast_features].copy()

print("Fast feature shape:", fast_features_df.shape)
print("\nMissing values:", fast_features_df.isnull().sum().sum())

In [ ]:

prompt_features = pd.get_dummies(
    df["prompt_name"],
    prefix="prompt"
)

print(prompt_features.head())
print(prompt_features.shape)

In [ ]:

# Features — the FAST feature set (no TF-IDF, no heavy NLP — that runs after the models)
X = pd.concat([df[["essay_id"]], fast_features_df], axis=1)

# Target variable (essay score)
y = df["score"]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

In [ ]:

from sklearn.model_selection import train_test_split

Xremain, Xtest, yremain, ytest = train_test_split(
    X, y, test_size=0.20, random_state=42)
Xtrain, Xval, ytrain, yval = train_test_split(
    Xremain, yremain, test_size=0.25, random_state=42)

print("Training:", Xtrain.shape)
print("Validation:", Xval.shape)
print("Testing:", Xtest.shape)

In [ ]:

prompt_train = prompt_features.loc[Xtrain.index]
prompt_val = prompt_features.loc[Xval.index]
prompt_test = prompt_features.loc[Xtest.index]

In [ ]:

final_train = pd.concat(
    [Xtrain.drop(columns=["essay_id"]), prompt_train],
    axis=1
)

final_val = pd.concat(
    [Xval.drop(columns=["essay_id"]), prompt_val],
    axis=1
)

final_test = pd.concat(
    [Xtest.drop(columns=["essay_id"]), prompt_test],
    axis=1
)

print("Final feature shape (train):", final_train.shape)

In [ ]:

X_train = final_train.copy()
X_val = final_val.copy()
X_test = final_test.copy()

In [ ]:

X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_val = X_val.replace([np.inf, -np.inf], np.nan)
X_test = X_test.replace([np.inf, -np.inf], np.nan)

In [ ]:

X_train = X_train.fillna(X_train.median())
X_val = X_val.fillna(X_train.median())
X_test = X_test.fillna(X_train.median())

print(X_train.shape)
print(type(X_train))

In [ ]:

print(np.isinf(X_train).sum().sum())
print(X_train.isnull().sum().sum())

In [ ]:

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [ ]:

from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, cohen_kappa_score

ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, ytrain)
y_pred_val = ridge.predict(X_val_scaled)
y_predicted = ridge.predict(X_test_scaled)

mse_val = mean_squared_error(yval, y_pred_val)
mae_val = mean_absolute_error(yval, y_pred_val)
r2_val = r2_score(yval, y_pred_val)

y_pred_val_rounded = np.round(y_pred_val).astype(int)
qwk_val = cohen_kappa_score(yval, y_pred_val_rounded, weights="quadratic")

print("Validation MSE:", mse_val)
print("Validation MAE:", mae_val)
print("Validation R²:", r2_val)
print("Validation QWK:", qwk_val)

mse = mean_squared_error(ytest, y_predicted)
mae = mean_absolute_error(ytest, y_predicted)
r2 = r2_score(ytest, y_predicted)

y_pred_rounded = np.round(y_predicted).astype(int)
qwk = cohen_kappa_score(ytest, y_pred_rounded, weights="quadratic")

print("MSE:", mse)
print("MAE:", mae)
print("R²:", r2)
print("QWK:", qwk)


coef_dict = dict(zip(X_train.columns, ridge.coef_))
print("Coefficient for word_count:", coef_dict["word_count"])

In [ ]:

comparison = pd.DataFrame({
    'Actual Score': ytest.values[:10],
    'Predicted Score': y_predicted[:10]})
print("\nSample Predictions (First 10 Essays):")
print(comparison.round(2))

In [ ]:

#Gradient Boosting
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, cohen_kappa_score

gb_model = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42)

gb_model.fit(X_train, ytrain)
y_pred_gb_val = gb_model.predict(X_val)
y_pred_gb = gb_model.predict(X_test)

mse_gb_val = mean_squared_error(yval, y_pred_gb_val)
mae_gb_val = mean_absolute_error(yval, y_pred_gb_val)
r2_gb_val = r2_score(yval, y_pred_gb_val)

y_pred_gb_val_rounded = np.round(y_pred_gb_val).astype(int)
qwk_gb_val = cohen_kappa_score(
    yval,
    y_pred_gb_val_rounded,
    weights="quadratic"
)

print("Validation MSE: ", mse_gb_val)
print("Validation Mean Absolute error: ", mae_gb_val)
print("Validation R square score: ", r2_gb_val)
print("Validation QWK: ", qwk_gb_val)


mse_gb = mean_squared_error(ytest, y_pred_gb)
mae_gb = mean_absolute_error(ytest, y_pred_gb)
r2_gb = r2_score(ytest, y_pred_gb)


y_pred_gb_rounded = np.round(y_pred_gb).astype(int)
qwk_gb = cohen_kappa_score(
    ytest,
    y_pred_gb_rounded,
    weights="quadratic"
)


print("MSE: ", mse_gb)
print("Mean Absolute error: ", mae_gb)
print("R square score: ", r2_gb)
print("QWK: ", qwk_gb)

In [ ]:

comparison_df = pd.DataFrame({
    'Actual Score': ytest.values[:10],
    'Predicted Score': y_pred_gb[:10]})
print("\nSample Predictions (First 10 Essays):")
print(comparison_df.round(2))

# 5. Feature importance — top 20 only, for readability
importances = pd.Series(gb_model.feature_importances_, index=X_train.columns)
importances = importances.sort_values(ascending=False).head(20)

plt.figure(figsize=(9, 5))
importances.plot(kind='bar', color='green', edgecolor='black')
plt.title("Top 20 Feature Importances in Essay Scoring", fontsize=12, fontweight='bold')
plt.xlabel("Feature", fontsize=10)
plt.ylabel("Importance Score", fontsize=10)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:

# Random Forest model
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)
# Ignore unnecessary warnings
import warnings
warnings.filterwarnings("ignore")

print("Libraries imported successfully!")

# Create baseline Random Forest model
rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

print("Random Forest model created!")

In [ ]:

rf_model.fit(X_train, ytrain)

print("Model training completed!")

In [ ]:

# Make predictions on test data
y_pred_val = rf_model.predict(X_val)
y_pred = rf_model.predict(X_test)

print(y_pred[:10])

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Calculate metrics
mse_val = mean_squared_error(yval, y_pred_val)
mae_val = mean_absolute_error(yval, y_pred_val)
r2_val = r2_score(yval, y_pred_val)

print("Validation Mean Squared Error:", mse_val)
print("Validation Mean Absolute Error:", mae_val)
print("Validation R² Score:", r2_val)

mse = mean_squared_error(ytest, y_pred)
mae = mean_absolute_error(ytest, y_pred)
r2 = r2_score(ytest, y_pred)

print("Mean Squared Error:", mse)
print("Mean Absolute Error:", mae)
print("R² Score:", r2)

In [ ]:

from sklearn.metrics import cohen_kappa_score

# Round predictions because essay scores are whole numbers
y_pred_val_rounded = np.round(y_pred_val).astype(int)
y_pred_rounded = np.round(y_pred).astype(int)

print(y_pred_rounded[:10])

# Calculate Quadratic Weighted Kappa
qwk_val = cohen_kappa_score(
    yval,
    y_pred_val_rounded,
    weights="quadratic")

print("Validation Quadratic Weighted Kappa (QWK):", qwk_val)

qwk = cohen_kappa_score(
    ytest,
    y_pred_rounded,
    weights="quadratic")

print("Quadratic Weighted Kappa (QWK):", qwk)

In [ ]:

comparison_rf = pd.DataFrame({
    'Actual Score': ytest.values[:10],
    'Predicted Score': y_pred[:10]})
print("\nSample Predictions (First 10 Essays):")
print(comparison_rf.round(2))

In [ ]:

# Get feature importance scores
feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf_model.feature_importances_})

# Sort from most important to least important
feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False)

# Display top 20 features
feature_importance.head(20)

In [ ]:

plt.figure(figsize=(10,6))

plt.barh(
    feature_importance.head(20)["Feature"],
    feature_importance.head(20)["Importance"])

plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Top 20 Random Forest Feature Importance")

plt.gca().invert_yaxis()

plt.show()

In [ ]:

from sklearn.model_selection import GridSearchCV
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 10, 20],
    "min_samples_leaf": [1, 2, 5]}

In [ ]:

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

models = {
    "RF_100_trees": RandomForestRegressor(
        n_estimators=100,
        max_depth=None,
        min_samples_leaf=1,
        random_state=42,
        n_jobs=-1),

    "RF_200_trees": RandomForestRegressor(
        n_estimators=200,
        max_depth=None,
        min_samples_leaf=1,
        random_state=42,
        n_jobs=-1),

    "RF_depth20_leaf2": RandomForestRegressor(
        n_estimators=100,
        max_depth=20,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1)}

In [ ]:

plt.figure(figsize=(10,8))

top20 = feature_importance.head(20)

plt.barh(top20["Feature"], top20["Importance"])

plt.xlabel("Importance Score")
plt.ylabel("Feature")
plt.title("Top 20 Most Important Features in the Random Forest Model")

plt.gca().invert_yaxis()

plt.show()

In [ ]:

results = pd.DataFrame({
    "Metric": [
        "Mean Squared Error (MSE)",
        "Mean Absolute Error (MAE)",
        "R² Score",
        "Quadratic Weighted Kappa (QWK)"
    ],
    "Value": [
        mse,
        mae,
        r2,
        qwk
    ]})

results

In [ ]:

from sklearn.metrics import accuracy_score

def evaluate(name, y_true, y_pred):
    y_pred_rounded_ = np.clip(np.round(y_pred), y_true.min(), y_true.max()).astype(int)
    return {
        "model": name,
        "MSE": mean_squared_error(y_true, y_pred),
        "MAE": mean_absolute_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
        "Accuracy": accuracy_score(y_true, y_pred_rounded_),
        "QWK": cohen_kappa_score(y_true, y_pred_rounded_, weights="quadratic"),
    }

all_results = [
    evaluate("Ridge", ytest, y_predicted),
    evaluate("GradientBoosting", ytest, y_pred_gb),
    evaluate("RandomForest", ytest, y_pred),
]

results_df = pd.DataFrame(all_results).sort_values("MSE").reset_index(drop=True)
print(results_df)

best_model_name = results_df.iloc[0]["model"]
print(f"\nBest model (lowest MSE): {best_model_name}")
print(results_df.iloc[0])

In [ ]:

!pip install -q spacy yake
!python -m spacy download en_core_web_sm -q
import spacy
# Only load the components this pipeline actually uses: "tagger" + "attribute_ruler" for POS tags,
# and "ner" for named entities. "parser" and "lemmatizer" are the two most expensive components in
# en_core_web_sm and neither is used anywhere below, so they're disabled.
nlp = spacy.load("en_core_web_sm", disable=["parser", "lemmatizer"])

In [ ]:

#POS + NER function — extracts both from a single spaCy Doc
def pos_and_ner_features(doc):
    counts = {
        "noun_count": 0, "verb_count": 0, "adjective_count": 0, "adverb_count": 0,
        "pronoun_count": 0, "determiner_count": 0, "preposition_count": 0, "conjunction_count": 0,
    }
    for token in doc:
        if token.pos_ == "NOUN":
            counts["noun_count"] += 1
        elif token.pos_ == "VERB":
            counts["verb_count"] += 1
        elif token.pos_ == "ADJ":
            counts["adjective_count"] += 1
        elif token.pos_ == "ADV":
            counts["adverb_count"] += 1
        elif token.pos_ == "PRON":
            counts["pronoun_count"] += 1
        elif token.pos_ == "DET":
            counts["determiner_count"] += 1
        elif token.pos_ == "ADP":
            counts["preposition_count"] += 1
        elif token.pos_ == "CCONJ":
            counts["conjunction_count"] += 1

    entities = doc.ents
    counts.update({
        "entity_count": len(entities),
        "person_count": sum(ent.label_ == "PERSON" for ent in entities),
        "organization_count": sum(ent.label_ == "ORG" for ent in entities),
        "location_count": sum(ent.label_ == "GPE" for ent in entities),
        "date_count": sum(ent.label_ == "DATE" for ent in entities),
    })
    return counts

In [ ]:

from tqdm.auto import tqdm

MAX_CHARS = 4000  # set to None to process full essays instead (slower, exact)

texts = df["full_text"].astype(str).tolist()
if MAX_CHARS is not None:
    texts = [t[:MAX_CHARS] for t in texts]

pos_ner_rows = [
    pos_and_ner_features(doc)
    for doc in tqdm(nlp.pipe(texts, batch_size=500, n_process=-1), total=len(texts), desc="POS + NER")
]
pos_df = pd.DataFrame(pos_ner_rows, index=df.index)

print(pos_df.head())
print(pos_df.shape)

df = pd.concat([df, pos_df], axis=1)

In [ ]:

pos_columns = [
    "noun_count",
    "verb_count",
    "adjective_count",
    "adverb_count",
    "pronoun_count",
    "determiner_count",
    "preposition_count",
    "conjunction_count"]

for col in pos_columns:
    df[col + "_ratio"] = (df[col] / df["word_count"].replace(0, np.nan))

df[[c + "_ratio" for c in pos_columns]] = df[[c + "_ratio" for c in pos_columns]].fillna(0)

In [ ]:

from nltk.tokenize import word_tokenize, sent_tokenize
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

#Lexical features
def lexical_features(text, window=50):
  #window(divides each section into 50 words)

    words = [w.lower()
        for w in word_tokenize(str(text))
        if w.isalpha()]


    if len(words) == 0:
        return pd.Series({
            "unique_word_count": 0,
            "lexical_ttr": 0,
            "mattr": 0})

    unique_words = len(set(words))
#ttr(type token ratio)
    ttr = unique_words / len(words)

    if len(words) < window:
        mattr = ttr
    else:
        ratios = []

        for i in range(len(words) - window + 1):

            section = words[i:i + window]

            ratios.append(len(set(section)) / window)
#mattr(moving average type token ratio)
        mattr = np.mean(ratios)

    return pd.Series({
        "unique_word_count": unique_words,
        "lexical_ttr": ttr,
        "mattr": mattr})

lexical_df = df["full_text"].apply(lexical_features)
print(lexical_df.head())
print(lexical_df.shape)

df = pd.concat([df, lexical_df], axis=1)

In [ ]:

#Keyword Extraction
import yake

keyword_extractor = yake.KeywordExtractor(
    lan="en",
    n=2,
    dedupLim=0.9,
    top=10)

def extract_keywords(text):

    keywords = keyword_extractor.extract_keywords(str(text))

    return [keyword for keyword, score in keywords]
sample_keywords = extract_keywords(df["full_text"].iloc[0])

print(sample_keywords)

def keyword_count(text):

    keywords = keyword_extractor.extract_keywords(str(text))

    return len(keywords)

In [ ]:
#Grammar
!pip install -q language-tool-python
import language_tool_python
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

tool = language_tool_python.LanguageTool("en-US")

sample = df["full_text"].iloc[0]
matches = tool.check(sample)
print("Errors:", len(matches))

def grammar_error_count(text):
    matches = tool.check(str(text))
    return len(matches)

GRAMMAR_SAMPLE_IDX = sample_idx

texts = df.loc[GRAMMAR_SAMPLE_IDX, "full_text"].astype(str).tolist()
with ThreadPoolExecutor(max_workers=8) as executor:
    grammar_counts = list(tqdm(executor.map(grammar_error_count, texts), total=len(texts), desc="Grammar check"))

df["grammar_errors"] = np.nan
df.loc[GRAMMAR_SAMPLE_IDX, "grammar_errors"] = grammar_counts

print("Grammar errors filled for:", df["grammar_errors"].notna().sum(), "of", len(df), "essays")

In [ ]:

linguistic_features = [
    "word_count",
    "sentence_count",
    "character_count",
    "paragraph_count",
    "avg_word_length",
    "avg_sentence_length",
    "unique_word_count",
    "type_token_ratio",
    "lexical_ttr",
    "mattr",
    "stopword_ratio",
    "flesch_reading_ease",
    "flesch_kincaid_grade",
    "gunning_fog_score",
    "dale_chall_readability_score",
    "guiraud_index",
    "long_word_ratio",
    "conjunction_ratio",

    "noun_count",
    "verb_count",
    "adjective_count",
    "adverb_count",
    "pronoun_count",
    "determiner_count",
    "preposition_count",
    "conjunction_count",

    "noun_count_ratio",
    "verb_count_ratio",
    "adjective_count_ratio",
    "adverb_count_ratio",
    "pronoun_count_ratio",
    "determiner_count_ratio",
    "preposition_count_ratio",
    "conjunction_count_ratio",

    "entity_count",
    "person_count",
    "organization_count",
    "location_count",
    "date_count",

    "grammar_errors"]

nlp_features = df[linguistic_features].copy()

print("NLP feature shape:", nlp_features.shape)
print("\nMissing values:", nlp_features.isnull().sum().sum())

In [ ]:

from transformers import BertTokenizer

#TOKENIZATION
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

sample_text = df["full_text"].str.lower().iloc[0]
tokens = tokenizer.tokenize(sample_text)
print("Tokens:", tokens)

token_ids = tokenizer.encode(sample_text, add_special_tokens=True)
print("Token IDs:", token_ids)

df["clean_text"] = df["full_text"].str.lower()
df["transformer_tokens"] = df["clean_text"].apply(lambda x: tokenizer.tokenize(x))

In [ ]:

nlp_blank = spacy.blank("en")
nlp_blank.add_pipe("sentencizer")
texts = df["clean_text"].tolist()
docs = list(nlp_blank.pipe(texts, batch_size=100))

df["sentences"] = [[sent.text.strip() for sent in doc.sents] for doc in docs]

print(df[["essay_id", "sentences"]].head())

In [ ]:

def clean_whitespace(text):
    text = text.strip()
    text = re.sub(r"\s+", " ", text)
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    return text

df["clean_text"] = df["clean_text"].apply(clean_whitespace)
print(df[["essay_id", "clean_text"]].head())

In [ ]:

def clean_text_final(text):
    text = text.lower()
    text = text.strip()
    text = re.sub(r"\s+", " ", text)
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    return text

def tokenize_text(text):
    doc = nlp_blank(text)
    return [token.text for token in doc]

def segment_sentences(text):
    doc = nlp_blank(text)
    return [sent.text.strip() for sent in doc.sents]

df["clean_text"] = df["clean_text"].apply(clean_text_final)
df["tokens"] = df["clean_text"].apply(tokenize_text)
df["sentences"] = df["clean_text"].apply(segment_sentences)

print(df[["essay_id", "clean_text", "tokens", "sentences"]].head())

In [ ]:

import random
import json

THRESHOLDS = {
    "word_count": {"low": 150, "high": 350},
    "avg_sentence_length": {"low": 10, "high": 25},
    "type_token_ratio": {"low": 0.35, "high": 0.55},
    "flesch_reading_ease": {"low": 30, "high": 70},
    "grammar_error_rate": {"low": 0.005, "high": 0.02},
    "spelling_error_rate": {"low": 0.002, "high": 0.01},
}

def load_thresholds_from_csv(csv_path, low_q=0.25, high_q=0.75):
    df_ = pd.read_csv(csv_path)
    computed = {}

    simple_features = ["word_count", "avg_sentence_length", "type_token_ratio", "flesch_reading_ease"]
    for feat in simple_features:
        if feat in df_.columns:
            computed[feat] = {
                "low": round(df_[feat].quantile(low_q), 3),
                "high": round(df_[feat].quantile(high_q), 3),
            }

    if "grammar_error_count" in df_.columns and "word_count" in df_.columns:
        rate = df_["grammar_error_count"] / df_["word_count"].replace(0, pd.NA)
        computed["grammar_error_rate"] = {
            "low": round(rate.quantile(low_q), 5),
            "high": round(rate.quantile(high_q), 5),
        }

    if "spelling_error_count" in df_.columns and "word_count" in df_.columns:
        rate = df_["spelling_error_count"] / df_["word_count"].replace(0, pd.NA)
        computed["spelling_error_rate"] = {
            "low": round(rate.quantile(low_q), 5),
            "high": round(rate.quantile(high_q), 5),
        }

    return computed

def _rate(count, word_count):
    if not word_count:
        return 0
    return count / word_count

In [ ]:

OVERALL_COMMENTS = {
    "high": [
        "This is a strong essay overall, showing clear command of the topic and writing conventions.",
        "Overall, this essay is well-executed, with clear ideas and confident use of language.",
        "This is a high-quality piece of writing that demonstrates strong control of the topic.",
    ],
    "good": [
        "This is a solid essay with clear strengths, alongside some areas that would benefit from revision.",
        "Overall this essay does a good job addressing the prompt, though a few areas could be sharpened.",
        "This essay shows good understanding of the topic, with some room to polish specific areas.",
    ],
    "developing": [
        "This essay shows a reasonable attempt but needs meaningful improvement in several areas.",
        "This is a developing piece of writing — the core ideas are present but need stronger execution.",
        "This essay has a foundation to build on, but several areas need attention before it's fully effective.",
    ],
    "low": [
        "This essay needs significant revision across structure, language, and development of ideas.",
        "This essay is not yet meeting the expectations of the task and needs substantial rework.",
        "Considerable revision is needed here, particularly around developing and organizing ideas.",
    ],
}

def _overall_comment(score, max_score, rng):
    pct = score / max_score if max_score else 0
    if pct >= 0.85:
        band = "high"
    elif pct >= 0.6:
        band = "good"
    elif pct >= 0.4:
        band = "developing"
    else:
        band = "low"
    return rng.choice(OVERALL_COMMENTS[band])

In [ ]:

def _check_length(data):
    wc = data.get("word_count", 0)
    t = THRESHOLDS["word_count"]
    if wc < t["low"]:
        return False, [
            f"The essay is quite short ({wc} words), which limits how fully ideas can be developed.",
            f"At {wc} words, the essay may not give the argument enough room to develop.",
        ], [
            "Aim to expand key points with more explanation, examples, or evidence to strengthen your argument.",
            "Try adding a supporting example or explaining your reasoning further in each paragraph.",
        ]
    elif wc > t["high"]:
        return True, [
            f"The essay is well-developed in length ({wc} words), giving room to explore ideas fully.",
            f"Good length ({wc} words) allows the essay to develop its points thoroughly.",
        ], None
    return None, None, None


def _check_sentence_variety(data):
    asl = data.get("avg_sentence_length", 0)
    t = THRESHOLDS["avg_sentence_length"]
    if asl < t["low"]:
        return False, [
            f"Sentences are quite short on average ({asl:.1f} words/sentence), which can make writing feel choppy.",
        ], [
            "Try combining related short sentences using conjunctions or subordinate clauses for better flow.",
        ]
    elif asl > t["high"]:
        return False, [
            f"Sentences are quite long on average ({asl:.1f} words/sentence), which can hurt clarity.",
        ], [
            "Break up long sentences into shorter ones to improve readability.",
        ]
    return True, [
        f"Sentence length is well balanced ({asl:.1f} words/sentence), supporting readability.",
        f"Sentences flow at a comfortable, readable length ({asl:.1f} words on average).",
    ], None


def _check_vocabulary(data):
    ttr = data.get("type_token_ratio", 0)
    t = THRESHOLDS["type_token_ratio"]
    if ttr < t["low"]:
        return False, [
            f"Vocabulary usage is fairly repetitive (diversity score: {ttr:.2f}).",
        ], [
            "Try varying your word choice — avoid reusing the same words repeatedly; use synonyms where appropriate.",
        ]
    elif ttr >= t["high"]:
        return True, [
            f"The essay shows strong vocabulary diversity (score: {ttr:.2f}).",
            f"Word choice is varied and precise throughout (diversity score: {ttr:.2f}).",
        ], None
    return None, None, None


def _check_readability(data):
    fre = data.get("flesch_reading_ease", 50)
    t = THRESHOLDS["flesch_reading_ease"]
    if fre < t["low"]:
        return False, [
            "The writing is quite dense and may be difficult to read.",
        ], [
            "Simplify complex sentence structures and consider shorter, clearer phrasing.",
        ]
    elif fre > t["high"]:
        return True, [
            "The essay is clear and easy to read.",
            "The writing flows smoothly and is easy for a reader to follow.",
        ], None
    return None, None, None


def _check_grammar(data):
    if "grammar_error_count" not in data:
        return None, None, None
    rate = _rate(data["grammar_error_count"], data.get("word_count", 1))
    t = THRESHOLDS["grammar_error_rate"]
    if rate > t["high"]:
        return False, [
            f"There are a notable number of grammar issues ({data['grammar_error_count']} detected).",
        ], [
            "Proofread carefully for subject-verb agreement, verb tense consistency, and sentence fragments.",
        ]
    elif rate < t["low"]:
        return True, [
            "Grammar is consistently strong throughout the essay.",
            "The essay is largely free of grammatical errors.",
        ], None
    return None, None, None


def _check_spelling(data):
    if "spelling_error_count" not in data:
        return None, None, None
    rate = _rate(data["spelling_error_count"], data.get("word_count", 1))
    t = THRESHOLDS["spelling_error_rate"]
    if rate > t["high"]:
        return False, [
            f"There are several spelling errors ({data['spelling_error_count']} detected).",
        ], [
            "Run a spellcheck pass and read the essay aloud to catch missed errors.",
        ]
    return None, None, None


CHECKS = [
    _check_length,
    _check_sentence_variety,
    _check_vocabulary,
    _check_readability,
    _check_grammar,
    _check_spelling,
]

In [ ]:

def generate_feedback(essay_analysis: dict, essay_id=None) -> dict:
    score = essay_analysis.get("predicted_score")
    max_score = essay_analysis.get("max_score", 6)

    seed = hash(essay_id) if essay_id is not None else None
    rng = random.Random(seed)

    strengths, weaknesses, suggestions = [], [], []

    for check in CHECKS:
        is_strength, messages, sugg_options = check(essay_analysis)
        if is_strength is True:
            strengths.append(rng.choice(messages))
        elif is_strength is False:
            weaknesses.append(rng.choice(messages))
            if sugg_options:
                suggestions.append(rng.choice(sugg_options))

    if not strengths:
        strengths.append("The essay engages with the assigned topic.")
    if not weaknesses:
        weaknesses.append("No major issues detected in the areas analyzed.")

    return {
        "score": score,
        "max_score": max_score,
        "overall_comment": _overall_comment(score, max_score, rng),
        "strengths": strengths,
        "weaknesses": weaknesses,
        "suggestions": suggestions,
    }

In [ ]:

test_cases = {
    "High scorer": {
        "predicted_score": 6, "max_score": 6, "word_count": 420,
        "sentence_count": 22, "avg_sentence_length": 19.1,
        "type_token_ratio": 0.61, "flesch_reading_ease": 72,
        "flesch_kincaid_grade": 8.0, "grammar_errors": 1, "spelling_error_count": 0
    },
    "Average scorer": {
        "predicted_score": 4, "max_score": 6, "word_count": 260,
        "sentence_count": 15, "avg_sentence_length": 17.3,
        "type_token_ratio": 0.45, "flesch_reading_ease": 55,
        "flesch_kincaid_grade": 9.5, "grammar_errors": 4, "spelling_error_count": 1
    },
    "Low scorer": {
        "predicted_score": 1, "max_score": 6, "word_count": 90,
        "sentence_count": 6, "avg_sentence_length": 8.5,
        "type_token_ratio": 0.30, "flesch_reading_ease": 25,
        "flesch_kincaid_grade": 12.0, "grammar_errors": 9, "spelling_error_count": 6
    },
}

for name, data in test_cases.items():
    print(name)
    result = generate_feedback(data, essay_id=name)
    print(json.dumps(result, indent=2))
    print()

In [ ]:

nlp_features.to_csv("linguistic_features.csv", index=False)